# Resolve Salesforce Job + Worksite for Kimedics rows

Order of resolution (same as agreed design):

1. **Cache — `job_current`** — if `sf_worksite_account_id` (and optionally `sf_job_id`) already set, use it.
2. **Cache — `job_content` history** — newest row first: reuse first non-null `sf_worksite_account_id` / `sf_job_id` for this `job_id`.
3. **Salesforce — practice match** — `norm(practice_value)` vs `norm(Job_Client_Job_Id__c)`; only if **exactly one** `Job__c` matches, take `Id` + `Job_Worksite_Location_1__c`.
4. **Unmapped** — no confident SF job; treat as “new post” path elsewhere (create worksite + job).

Optional: add `sf_job_id` column, then apply updates with `DRY_RUN = False`.

In [8]:
import os, sys, re
from pathlib import Path
from collections import defaultdict

project_root = Path.cwd().resolve()
for _ in range(15):
    if (project_root / "src" / "utils").is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError("Could not locate repo root containing src/utils")

sys.path.insert(0, str(project_root / "src"))
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

SCHEMA = "public"  # "public" | "staging"
DRY_RUN = False
RUN_SCHEMA_MIGRATE = True  # adds sf_job_id + other extra columns via supabase_db.migrate

if SCHEMA not in ("public", "staging"):
    raise ValueError("SCHEMA must be 'public' or 'staging'")

print("Project root:", project_root)
print("SCHEMA:", SCHEMA, "  DRY_RUN:", DRY_RUN)

Project root: /Users/andylee/Desktop/projects/proxi/proxi_salesforce_automation
SCHEMA: public   DRY_RUN: False


## 1. Ensure DB columns (`sf_job_id`, etc.)

Uses `migrate_job_tables_extra_columns` from `utils.supabase_db` (same as production). Safe to re-run.

In [9]:
from utils.supabase_db import get_conn, ensure_tables, ensure_full_staging_schema

if not RUN_SCHEMA_MIGRATE:
    print("Skipping schema migrate.")
elif DRY_RUN:
    print("DRY_RUN: would run ensure_tables / ensure_full_staging_schema (adds sf_job_id via migrate)")
else:
    with get_conn() as conn:
        if SCHEMA == "staging":
            ensure_full_staging_schema(conn, SCHEMA)
        else:
            ensure_tables(conn)
    print("Schema ready (includes sf_job_id on job_current / job_content when migrated).")

Schema ready (includes sf_job_id on job_current / job_content when migrated).


## 2. Load Salesforce jobs + practice index (`kimedics_sf_mapping` logic)

In [10]:
def norm(val):
    s = (val or "").strip().lower()
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"\s*-\s*(closed|closing)\s*$", "", s)
    s = re.sub(r"[,.\-–]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

from utils.salesforce import pull_all_jobs

TOKEN_URL = os.environ.get("SALESFORCE_TOKEN_URL") or "https://proxi.my.salesforce.com"
USE_SANDBOX = os.environ.get("SALESFORCE_USE_SANDBOX", "").lower() in ("1", "true", "yes")
USE_CC = os.environ.get("SALESFORCE_USE_USERNAME_PASSWORD", "").lower() not in ("1", "true", "yes")

sf_jobs = pull_all_jobs(
    consumer_key=os.environ["SALESFORCE_CONSUMER_KEY"],
    consumer_secret=os.environ["SALESFORCE_CONSUMER_SECRET"],
    username=os.environ.get("SALESFORCE_USERNAME") or None,
    password=os.environ.get("SALESFORCE_PASSWORD") or None,
    use_client_credentials=USE_CC,
    token_url=TOKEN_URL,
    security_token=os.environ.get("SALESFORCE_SECURITY_TOKEN") or None,
    use_sandbox=USE_SANDBOX,
)
sf_by_id = {r["Id"]: r for r in sf_jobs}
sf_by_practice = defaultdict(set)
for r in sf_jobs:
    p = norm(r.get("Job_Client_Job_Id__c"))
    if p:
        sf_by_practice[p].add(r["Id"])

print(f"SF jobs: {len(sf_jobs)}  practice keys: {len(sf_by_practice)}")

SF jobs: 4457  practice keys: 812


## 3. Load `job_current` + helpers for `job_content` lookup

In [11]:
from utils.supabase_db import get_job_current, get_job_content

with get_conn() as conn:
    kim_rows = get_job_current(conn, limit=None, schema=SCHEMA)
print(f"job_current rows: {len(kim_rows)}")


def _strip_ids(r):
    wid = (r.get("sf_worksite_account_id") or "").strip()
    jid = (r.get("sf_job_id") or "").strip() if r.get("sf_job_id") is not None else ""
    return wid, jid


def latest_sf_from_job_content(conn, job_id: str):
    """Newest job_content first; return first row with any SF id filled."""
    rows = get_job_content(conn, limit=500, schema=SCHEMA, job_ids=[job_id])
    for r in rows:
        wid, jid = _strip_ids(r)
        if wid or jid:
            return {"sf_worksite_account_id": wid or None, "sf_job_id": jid or None, "source": "job_content", "job_content_id": r.get("id")}
    return None


def resolve_row(conn, kr: dict) -> dict:
    """Returns desired sf ids for one Kimedics row (fills missing pieces)."""
    job_id = str(kr.get("job_id") or "").strip()
    out = {
        "job_id": job_id,
        "resolution": "",
        "sf_job_id": "",
        "sf_worksite_account_id": "",
        "detail": "",
    }

    w_cur, j_cur = _strip_ids(kr)
    have_w = bool(w_cur)
    have_j = bool(j_cur)

    # If both already present, nothing to do.
    if have_w and have_j:
        out["resolution"] = "job_current_cache"
        out["sf_worksite_account_id"] = w_cur
        out["sf_job_id"] = j_cur
        out["detail"] = "already on job_current"
        return out

    # Try to fill missing pieces from job_content history.
    prev = latest_sf_from_job_content(conn, job_id)
    w, j = w_cur, j_cur
    if prev:
        w_prev = (prev.get("sf_worksite_account_id") or "").strip()
        j_prev = (prev.get("sf_job_id") or "").strip()
        w = w_cur or w_prev
        j = j_cur or j_prev
        if w or j:
            out["resolution"] = "job_content_cache" if not (have_w or have_j) else "job_current_partial+job_content"
            out["sf_worksite_account_id"] = w
            out["sf_job_id"] = j
            out["detail"] = f"filled from job_content id={prev.get('job_content_id')}"
            # If we still don't have both IDs, continue to SF match below.
            if w and j:
                return out

    # Practice-only match to Salesforce to fill remaining missing pieces.
    p = norm(kr.get("practice_value"))
    if not p:
        out["resolution"] = "unmapped"
        out["detail"] = "missing practice_value"
        return out

    hits = sorted(sf_by_practice.get(p, set()))
    if len(hits) == 0:
        out["resolution"] = "unmapped"
        out["detail"] = "no SF Job_Client_Job_Id__c match"
        return out
    if len(hits) > 1:
        out["resolution"] = "unmapped"
        out["detail"] = f"ambiguous practice: {len(hits)} SF jobs"
        return out

    sid = hits[0]
    sf = sf_by_id.get(sid) or {}
    wid = (sf.get("Job_Worksite_Location_1__c") or "").strip()

    # If we already pulled something from job_content above, keep that and only fill missing pieces.
    if out["resolution"] in ("job_content_cache", "job_current_partial+job_content"):
        out["resolution"] = out["resolution"] + "+sf_practice_match"
    else:
        out["resolution"] = "sf_practice_match" if not (have_w or have_j) else "job_current_partial+sf_practice_match"

    out["sf_job_id"] = j or sid
    out["sf_worksite_account_id"] = w or wid
    out["detail"] = "1:1 practice match"
    return out

job_current rows: 111


## 4. Run resolution for every `job_current` row + summary

In [12]:
import pandas as pd

results = []
with get_conn() as conn:
    for kr in kim_rows:
        results.append(resolve_row(conn, kr))

df = pd.DataFrame(results)
print(df["resolution"].value_counts().to_string())
print("\nUnmapped sample:")
display(df[df["resolution"] == "unmapped"].head(20))

resolution
job_current_partial+job_content+sf_practice_match    104
unmapped                                               7

Unmapped sample:


,job_id,resolution,sf_job_id,sf_worksite_account_id,detail
15,19553,unmapped,,,no SF Job_Client_Job_Id__c match
27,19531,unmapped,,,no SF Job_Client_Job_Id__c match
47,19437,unmapped,,,no SF Job_Client_Job_Id__c match
54,19453,unmapped,,,no SF Job_Client_Job_Id__c match
58,19455,unmapped,,,no SF Job_Client_Job_Id__c match
85,19488,unmapped,,,no SF Job_Client_Job_Id__c match
108,19523,unmapped,,,no SF Job_Client_Job_Id__c match


## 5. Apply to Supabase (only rows that need new values)

Updates `job_current` and **all** `job_content` rows for the same `job_id` when resolution is `sf_practice_match` or `job_content_cache` (copying forward from content to current is handled by apply only when you want to persist practice matches; cache hits on `job_current` are skipped).

Set `DRY_RUN = False` in the first cell to write.

In [13]:
# Apply any row where our resolved sf ids differ from what job_current currently has.

kim_by_job = {str(r.get("job_id") or "").strip(): r for r in kim_rows}

candidates = df[df["resolution"].ne("unmapped")].copy()

def needs_write(row):
    kr = kim_by_job.get(row["job_id"]) or {}
    w0, j0 = _strip_ids(kr)
    w1 = (row.get("sf_worksite_account_id") or "").strip()
    j1 = (row.get("sf_job_id") or "").strip()
    return bool((w1 and w1 != w0) or (j1 and j1 != j0))

to_apply = candidates[candidates.apply(needs_write, axis=1)].copy()
print(f"Rows to apply: {len(to_apply)}")

if DRY_RUN:
    print("DRY_RUN: no UPDATE executed")
    display(to_apply.head(20))
else:
    with get_conn() as conn:
        with conn.cursor() as cur:
            for _, row in to_apply.iterrows():
                jid = row["job_id"]
                wid = (row.get("sf_worksite_account_id") or "").strip() or None
                sfjid = (row.get("sf_job_id") or "").strip() or None
                cur.execute(
                    f'UPDATE "{SCHEMA}".job_current '
                    f'SET sf_worksite_account_id = COALESCE(%s, sf_worksite_account_id), '
                    f'    sf_job_id = COALESCE(%s, sf_job_id) '
                    f'WHERE job_id = %s;',
                    (wid, sfjid, jid),
                )
                cur.execute(
                    f'UPDATE "{SCHEMA}".job_content '
                    f'SET sf_worksite_account_id = COALESCE(%s, sf_worksite_account_id), '
                    f'    sf_job_id = COALESCE(%s, sf_job_id) '
                    f'WHERE job_id = %s;',
                    (wid, sfjid, jid),
                )
        conn.commit()
    print("Committed updates.")

Rows to apply: 104
Committed updates.


## 6. Export CSV (optional)

In [14]:
out_dir = project_root / "data" / "exports"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"sf_job_worksite_resolve_{SCHEMA}.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

Wrote /Users/andylee/Desktop/projects/proxi/proxi_salesforce_automation/data/exports/sf_job_worksite_resolve_public.csv
